

*   harmony (two or more notes sounding together to build chords and add depth),
*   melody (a sequence of pitches and rhythms that form a recognizable tune),
*   rhythm (the pattern and placement of sounds in time).





Cell 1 — Setup & Helper Functions

In [1]:
import numpy as np
from IPython.display import Audio, display

SR = 44100  # Sample rate (samples per second)

# Equal temperament: MIDI note -> frequency in Hz
def note_to_freq(midi_note):
    return 440.0 * (2 ** ((midi_note - 69) / 12))

# Generate a sine tone with an ADSR envelope (so notes don't click)
def tone(freq, duration, amp=0.5, sr=SR):
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    wave = np.sin(2 * np.pi * freq * t)

    # Attack-Decay-Sustain-Release envelope
    n = len(wave)
    a, d, r = int(0.01 * sr), int(0.1 * sr), int(0.2 * sr)
    s = max(n - a - d - r, 0)
    env = np.concatenate([
        np.linspace(0, 1, a),
        np.linspace(1, 0.7, d),
        np.full(s, 0.7),
        np.linspace(0.7, 0, r)
    ])[:n]
    return amp * wave * env

# Mix a list of (freq, duration) events sequentially
def sequence(events, amp=0.4):
    return np.concatenate([tone(f, d, amp) for f, d in events])

# Mix several voices together (same length), normalized
def mix(voices):
    length = max(len(v) for v in voices)
    out = np.zeros(length)
    for v in voices:
        out[:len(v)] += v
    return out / max(1, np.max(np.abs(out)))

Cell 2 — 🎼 Harmony (chords: notes sounding together)

In [2]:
# A major triad: C4 (60), E4 (64), G4 (67)
chord_notes = [60, 64, 67]
chord = mix([tone(note_to_freq(n), 1.5, amp=0.4) for n in chord_notes])
print("🎼 Harmony — C major triad (C-E-G played simultaneously)")
display(Audio(chord, rate=SR))

# Try a chord progression: C -> Am -> F -> G
progression = [
    [60, 64, 67],  # C major
    [57, 60, 64],  # A minor
    [53, 57, 60],  # F major
    [55, 59, 62],  # G major
]
prog_audio = np.concatenate([
    mix([tone(note_to_freq(n), 0.8, amp=0.4) for n in chord])
    for chord in progression
])
print("🎼 Harmony — I–vi–IV–V progression")
display(Audio(prog_audio, rate=SR))

🎼 Harmony — C major triad (C-E-G played simultaneously)


🎼 Harmony — I–vi–IV–V progression


Cell 3 — 🎶 Melody (pitches + rhythm forming a tune)

In [3]:
# Simple melody: (MIDI note, duration in seconds)
# "Twinkle Twinkle Little Star" opening phrase
melody = [
    (60, 0.5), (60, 0.5), (67, 0.5), (67, 0.5),
    (69, 0.5), (69, 0.5), (67, 1.0),
    (65, 0.5), (65, 0.5), (64, 0.5), (64, 0.5),
    (62, 0.5), (62, 0.5), (60, 1.0),
]

events = [(note_to_freq(n), d) for n, d in melody]
melody_audio = sequence(events, amp=0.4)

print("🎶 Melody — Twinkle Twinkle Little Star")
display(Audio(melody_audio, rate=SR))

🎶 Melody — Twinkle Twinkle Little Star


Cell 4 — 🥁 Rhythm (pattern & placement of sounds in time)

In [4]:
def kick(duration=0.2, sr=SR):
    """Low sine sweep — sounds like a kick drum."""
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    freq = np.linspace(150, 50, len(t))
    return 0.9 * np.sin(2 * np.pi * freq * t) * np.exp(-8 * t)

def snare(duration=0.15, sr=SR):
    """Filtered noise burst — sounds like a snare/clap."""
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    noise = np.random.randn(len(t))
    return 0.4 * noise * np.exp(-20 * t)

def hat(duration=0.05, sr=SR):
    """Short high-frequency noise — hi-hat."""
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    noise = np.random.randn(len(t))
    return 0.25 * noise * np.exp(-60 * t)

# 16-step pattern @ 120 BPM (each step = 0.125s => 8th notes at 120bpm~.25s)
step_dur = 0.125
steps = 16

# Patterns: 1 = hit, 0 = rest
kick_pat  = [1,0,0,0, 0,0,1,0, 0,0,1,0, 0,0,0,0]
snare_pat = [0,0,0,0, 1,0,0,0, 0,0,0,0, 1,0,0,0]
hat_pat   = [1,0,1,0, 1,0,1,0, 1,0,1,0, 1,0,1,0]

def render_pattern(pattern, sound_fn):
    track = np.zeros(int(SR * step_dur * steps))
    for i, hit in enumerate(pattern):
        if hit:
            start = int(i * step_dur * SR)
            s = sound_fn()
            end = min(start + len(s), len(track))
            track[start:end] += s[:end - start]
    return track

beat = render_pattern(kick_pat, kick) \
     + render_pattern(snare_pat, snare) \
     + render_pattern(hat_pat, hat)

beat = beat / np.max(np.abs(beat)) * 0.9
print("🥁 Rhythm — basic 16-step drum pattern (kick / snare / hi-hat)")
display(Audio(beat, rate=SR))

🥁 Rhythm — basic 16-step drum pattern (kick / snare / hi-hat)


Cell 5 — 🎧 Put it all together: Melody + Harmony + Rhythm

In [5]:
# Layer a melody on top of the drum beat with chords underneath.

# 1. Rhythm (length = 16 * 0.125 = 2.0s). Repeat 4x for ~8s
beat_loop = np.tile(beat, 4)

# 2. Harmony: I–vi–IV–V, each 2s, aligned with beat
chord_prog = [
    [60, 64, 67], [57, 60, 64], [53, 57, 60], [55, 59, 62]
]
harmony_audio = np.concatenate([
    mix([tone(note_to_freq(n), 2.0, amp=0.25) for n in chord])
    for chord in chord_prog
])

# 3. Melody on top (reuse Twinkle, played twice)
melody_loop = np.tile(melody_audio, 2)

# Align all to same length
L = max(len(beat_loop), len(harmony_audio), len(melody_loop))
def pad(x): return np.pad(x, (0, L - len(x)))

final = pad(beat_loop) * 0.8 + pad(harmony_audio) * 0.9 + pad(melody_loop) * 0.7
final = final / np.max(np.abs(final)) * 0.95

print("🎧 Full mix: rhythm + harmony + melody")
display(Audio(final, rate=SR))

🎧 Full mix: rhythm + harmony + melody
